# Comparativa de Modelos de Predicción de Churn
En este notebook presentaremos una comparativa estructurada entre un modelo base estándar (**Regresión Logística**) y un modelo avanzado basado en árboles de decisión (**XGBoost**).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score

# Importar XGBoost
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

## 1. Carga y Preprocesamiento de Datos
Ambos modelos usarán exactamente los mismos datos para garantizar una comparativa justa.

In [ ]:
# Cargar datos
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Limpieza básica
if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Separar X y y
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

# One-Hot Encoding
categorical_cols = X.select_dtypes(include=['object']).columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Escalar (Necesario para Regresión Logística)
numerical_cols = X.select_dtypes(exclude=['object', 'bool']).columns
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# Renombrar columnas para evitar errores con JSON/XGBoost que no soporta ciertos caracteres
import re
regex = re.compile(r"\[|\]|<", re.IGNORECASE)
X.columns = [regex.sub("_", col) if any(x in str(col) for x in set('[ ] <')) else col for col in X.columns.values]

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Datos preparados con éxito.")

## 2. Modelo Baseline: Regresión Logística
La Regresión Logística es el estándar estadístico para problemas de clasificación binaria. Actuará como nuestro punto de referencia.

In [ ]:
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)

# Predicciones
y_pred_base = baseline_model.predict(X_test)
y_prob_base = baseline_model.predict_proba(X_test)[:, 1]

print("--- REPORTE BASELINE (Regresión Logística) ---")
print(classification_report(y_test, y_pred_base))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_base), annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión - Baseline')
plt.ylabel('Real')
plt.xlabel('Predicción')
plt.show()

## 3. Modelo Optimizado: XGBoost con Balanceo de Clases
XGBoost es un potente algoritmo de Gradient Boosting. Utilizaremos el parámetro `scale_pos_weight` para darle mayor importancia a la clase minoritaria (Churn = Yes), penalizando fuertemente los errores de falsos negativos.

In [ ]:
# Calcular el peso para la clase positiva (Churn)
# Fórmula típica: N_Negativos / N_Positivos
ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)
print(f"Ratio de desbalance (Negativos/Positivos): {ratio:.2f}")

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=ratio, 
    random_state=42, 
    eval_metric='logloss',
    max_depth=4,         # Limitamos profundidad para evitar sobreajuste
    learning_rate=0.05   # Aprendizaje más lento y estable
)
xgb_model.fit(X_train, y_train)

# Predicciones
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("--- REPORTE MODELO OPTIMIZADO (XGBoost) ---")
print(classification_report(y_test, y_pred_xgb))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_xgb), annot=True, fmt='d', cmap='Greens')
plt.title('Matriz de Confusión - XGBoost')
plt.ylabel('Real')
plt.xlabel('Predicción')
plt.show()

## 4. Comparativa Final Explicativa
Analizaremos el rendimiento lado a lado mediante una Curva ROC (que mide la capacidad de separar ambas clases independientemente del umbral) y una tabla resumen centrada en el **Recall** (Nuestra capacidad para detectar desertores).

In [ ]:
# --- 1. Gráfica Curva ROC ---
fpr_base, tpr_base, _ = roc_curve(y_test, y_prob_base)
auc_base = auc(fpr_base, tpr_base)

fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
auc_xgb = auc(fpr_xgb, tpr_xgb)

plt.figure(figsize=(8, 6))
plt.plot(fpr_base, tpr_base, color='blue', lw=2, label=f'Baseline (AUC = {auc_base:.3f})')
plt.plot(fpr_xgb, tpr_xgb, color='green', lw=2, label=f'XGBoost (AUC = {auc_xgb:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos (Recall)')
plt.title('Curva ROC: Baseline vs XGBoost')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

# --- 2. Tabla de Métricas ---
def get_metrics(y_true, y_pred):
    return {
        'Accuracy (Global)': f"{accuracy_score(y_true, y_pred):.1%}",
        'Precision (Churn)': f"{precision_score(y_true, y_pred):.1%}",
        'Recall (Churn)': f"{recall_score(y_true, y_pred):.1%}",
        'F1-Score (Churn)': f"{f1_score(y_true, y_pred):.1%}"
    }

metrics_df = pd.DataFrame({
    'Baseline (Regresión Logística)': get_metrics(y_test, y_pred_base),
    'Optimizado (XGBoost)': get_metrics(y_test, y_pred_xgb)
}).T

print("\n=== RESUMEN COMPARATIVO ===")
display(metrics_df)

## 5. Exportar el Modelo XGBoost
Vamos a guardar el modelo ganador (XGBoost) y el escalador para usarlos en producción local.

In [ ]:
import joblib

# 1. Guardar modelo XGBoost 
joblib.dump(xgb_model, 'xgboost_churn_model.pkl')

# 2. Guardar el escalador
joblib.dump(scaler, 'scaler_xgb.pkl')

# 3. Guardar nombres de columnas exactos
joblib.dump(list(X.columns), 'model_columns_xgb.pkl')

print("¡Archivos del modelo XGBoost guardados correctamente!")
print("Descárgalos desde el panel izquierdo.")